In [5]:
from sklearn.datasets import fetch_openml

credit_data = fetch_openml(
    name="credit",
    version=1,
    as_frame=True,
    parser="auto"
)

df = credit_data.data.copy()
target = credit_data.target.copy()

print("Base cargada para revisión")

Base cargada para revisión


In [4]:
print(df.shape)
print(df.head())
print(df.info())
print(target.head())
print(target.unique())
print(target.dtype)
print(target.value_counts())

(16714, 10)
   RevolvingUtilizationOfUnsecuredLines   age  \
0                              0.006999  38.0   
1                              0.704592  63.0   
2                              0.063113  57.0   
3                              0.368397  68.0   
4                              1.000000  34.0   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                                   0.0   0.302150         5440.0   
1                                   0.0   0.471441         8000.0   
2                                   0.0   0.068586         5000.0   
3                                   0.0   0.296273         6250.0   
4                                   1.0   0.000000         3500.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90DaysLate  \
0                              4.0                      0.0   
1                              9.0                      0.0   
2                             17.0                      0.0   
3                         

In [ ]:
# Revisión de duplicados
duplicados = df.duplicated().sum()
print(f"Cantidad de filas duplicadas: {duplicados}")

Cantidad de filas duplicadas: 2


In [ ]:
# Estadísticos descriptivos
df.describe().T

,count,mean,std,min,25%,50%,75%,max
RevolvingUtilizationOfUnsecuredLines,16714.0,4.799862,204.062345,0.0,0.082397,0.443080,0.926637,22000.0
age,16714.0,48.798672,13.906078,21.0,38.000000,48.000000,58.000000,101.0
NumberOfTime30-59DaysPastDueNotWorse,16714.0,1.110267,7.172890,0.0,0.000000,0.000000,1.000000,98.0
DebtRatio,16714.0,30.980298,719.694859,0.0,0.155971,0.322299,0.533426,61106.5
MonthlyIncome,16714.0,6118.120258,5931.841779,0.0,3128.500000,5000.000000,7573.000000,250000.0
NumberOfOpenCreditLinesAndLoans,16714.0,8.503709,5.370965,0.0,5.000000,8.000000,11.000000,57.0
NumberOfTimes90DaysLate,16714.0,0.863827,7.167576,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,16714.0,1.047445,1.272565,0.0,0.000000,1.000000,2.000000,29.0
NumberOfTime60-89DaysPastDueNotWorse,16714.0,0.734354,7.138737,0.0,0.000000,0.000000,0.000000,98.0
NumberOfDependents,16714.0,0.944358,1.198791,0.0,0.000000,0.000000,2.000000,8.0


In [14]:
# Percentiles sólo de variables predictoras
df.quantile([0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).T

,0.01,0.25,0.50,0.75,0.95,0.99
RevolvingUtilizationOfUnsecuredLines,0.0,0.082397,0.443080,0.926637,1.027668,1.447575
age,24.0,38.000000,48.000000,58.000000,74.000000,84.000000
NumberOfTime30-59DaysPastDueNotWorse,0.0,0.000000,0.000000,1.000000,3.000000,6.000000
DebtRatio,0.0,0.155971,0.322299,0.533426,1.239364,367.620000
MonthlyIncome,0.0,3128.500000,5000.000000,7573.000000,14000.000000,24166.000000
NumberOfOpenCreditLinesAndLoans,0.0,5.000000,8.000000,11.000000,18.000000,25.000000
NumberOfTimes90DaysLate,0.0,0.000000,0.000000,0.000000,2.000000,6.000000
NumberRealEstateLoansOrLines,0.0,0.000000,1.000000,2.000000,3.000000,5.000000
NumberOfTime60-89DaysPastDueNotWorse,0.0,0.000000,0.000000,0.000000,1.000000,4.000000
NumberOfDependents,0.0,0.000000,0.000000,2.000000,3.000000,4.000000


### Conclusiones del análisis exploratorio

El dataset contiene 16.714 observaciones y 10 variables predictoras numéricas. La variable objetivo corresponde a `SeriousDlqin2yrs`, codificada como binaria con clases 0 y 1. Las clases se encuentran perfectamente balanceadas, por lo que no se requiere aplicar técnicas de remuestreo.

No se identificaron valores nulos en las variables predictoras. Se detectaron 2 filas duplicadas, por lo que serán eliminadas antes del modelado.

El análisis de percentiles evidencia la presencia de valores extremos en variables como `DebtRatio`, `MonthlyIncome`, `RevolvingUtilizationOfUnsecuredLines` y variables asociadas a morosidad. Por esta razón, se implementará una estrategia de clipping al percentil 99 dentro del pipeline de preprocesamiento, asegurando que los límites se calculen exclusivamente sobre el conjunto de entrenamiento para evitar fuga de información.

Dado que todas las variables predictoras son numéricas, no será necesario aplicar codificación de variables categóricas. El escalamiento se utilizará sólo en el modelo lineal regularizado, mientras que los modelos basados en árboles se entrenarán sin escalamiento.